In [11]:
origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/'
import sys
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')

In [12]:
import os,cv2
os.chdir(origin+'Tokyo_Jobs\\Data_Collection\\Extract_Data')

from Split import VertPart
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart
from Read import Vision
from google.cloud import vision
from google.cloud.vision_v1 import types

In [13]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [14]:
import sys, json, os, cv2
import pandas as pd

Year,Showa='1942','17'
Quality='HQ' #HQ or Line#

df = pd.read_csv(origin+'Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)



In [15]:
StepAError,StepBError,MainError=[],[],[]
Dict={}
PosiDict={}
for Page in range(10, 80, 5):
    #Step A
    try:        
        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
        img=cv2.imread(os.path.join(origin,file_path))

        #Convert to book page
        BookPage=2*Page-14
        Rows=df[(df['Page']==BookPage)]
        NxRows=df[(df['Page']==BookPage+1)]
        if len(Rows)==0:
            print('No Office at Right Side. Page'+str(BookPage))
        if len(NxRows)==0:
            print('No Office at Left Side. Page'+str(BookPage+1))

        texts=Vision(img,'zh',client)
    except:
        StepAError.append(Page)
        
    #Step B
    try:
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')
        from Split import VertPart

        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from Organize import Filter, ConvertDict
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
        from PageCut import HoriPart

        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
        path=os.path.join(origin,file_path)

        HoriPoint=HoriPart(img,Page,texts)[0]

        try:
            VertPoint=VertPart(path)[1]
            print('Failed detecting Vertical Point')
            HH,WW=img.shape[:2]
            VertPoint=1*HH//3
        except:
            print('Failed detecting Vertical Point')
            HH,WW=img.shape[:2]
            VertPoint=HH//2

        ##Right Page
        BoxR=Filter(BookPage,texts,HoriPoint)
        BoxR=ConvertDict(BoxR)

        ##Left Page
        BoxL=Filter(BookPage+1,texts,HoriPoint)
        BoxL=ConvertDict(BoxL)
        
        Dict={}
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from Organize import FilterBox
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from DetectOffice import DetectOffice
        LocList=[]

        #RightPage
        OfficeList=Rows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxR, Office,VertPoint))
            BoxR=FilterBox(BoxR,LocList,VertPoint)

        #LeftPage
        OfficeList=NxRows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxL, Office,VertPoint))
            BoxL=FilterBox(BoxL,LocList,VertPoint)

        Dict[Page]=LocList
    except:
        StepBError.append(Page)
        continue
        
    # Main Code 
    try:        
        #Split quardrant into offices and detect Positions
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Position')
        from OrganizePosi import Split, SplitClova, Crop, CropClova
        from DetectPosi import DetectPosi,DetectPosiClova,AggregatePosi

        CrossWalk=pd.read_csv(origin+"Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
        Positions=CrossWalk.tolist()
        PosiDict['Pre'+str(Page)]=[]
        PosiDict_CLOVA['Pre'+str(Page)]=[]

        DF=Crop(texts,HoriPoint,VertPoint,Dict,Page)
        DF_CLOVA=CropClova(textsCLOVA,HoriPoint,VertPoint,Dict,Page)

        ##UR
        BoxList,BoxListCLOVA=Split(DF['UR']['Box'],DF['UR']['Thres']),SplitClova(DF_CLOVA['UR']['Box'],DF_CLOVA['UR']['Thres'])
        PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UR']['Thres'],PosiDict_CLOVA,Positions)

        ##LR
        BoxList,BoxListCLOVA=Split(DF['LR']['Box'],DF['LR']['Thres']),SplitClova(DF_CLOVA['LR']['Box'],DF_CLOVA['LR']['Thres'])
        PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LR']['Thres'],PosiDict_CLOVA,Positions)

        ##UL
        BoxList,BoxListCLOVA=Split(DF['UL']['Box'][1:],DF['UL']['Thres']),SplitClova(DF_CLOVA['UL']['Box'],DF_CLOVA['UL']['Thres'])
        PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UL']['Thres'],PosiDict_CLOVA,Positions)

        #LL
        BoxList,BoxListCLOVA=Split(DF['LL']['Box'],DF['LL']['Thres']),SplitClova(DF_CLOVA['LL']['Box'],DF_CLOVA['LL']['Thres'])
        PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LL']['Thres'],PosiDict_CLOVA,Positions)

        PosiDict=AggregatePosi(PosiDict,PosiDict_CLOVA)
    except:
        MainError.append(Page)

Failed detecting Vertical Point
Failed detecting Vertical Point
Failed detecting Vertical Point
No Office at Right Side. Page36
Failed detecting Vertical Point
No Office at Right Side. Page46
No Office at Left Side. Page47
Horizontal Line was not automatically detected... Defining line arbitrariry...
Failed detecting Vertical Point
Horizontal Line was not automatically detected... Defining line arbitrariry...
Failed detecting Vertical Point
Failed detecting Vertical Point
Failed detecting Vertical Point
No Office at Left Side. Page87
Failed detecting Vertical Point
Failed detecting Vertical Point
No Office at Right Side. Page106
No Office at Left Side. Page107
Failed detecting Vertical Point
No Office at Right Side. Page116
No Office at Left Side. Page117
Failed detecting Vertical Point
Failed detecting Vertical Point
Failed detecting Vertical Point


# Summary of performance

**1. Non-Running Error**

In [6]:
from ShowPosi import Show

def ErrorRate(ErrorList,PageList):
    return(round(len(ErrorList)/len(range(10, 120, 5)),2))

In [9]:
ErrorRate(StepAError,range(10, 120, 5)),ErrorRate(StepBError,range(10, 120, 5)),ErrorRate(MainError,range(10, 120, 5))

(0.64, 0.64, 0.0)

**2.Miss Assignment Error**

In [ ]:
for Office in list(PosiDict.keys()):
    print(Office)
    if 'Pre' in Office:
        continue
    try:
        BookPage=df[df['Office']==Office]['Page'].tolist()[0]
    except:
        continue
    
    if BookPage%2==0:
        Page=(BookPage+14)//2
    else:
        Page=(BookPage+13)//2

    file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
    path=os.path.join(origin,file_path)
    
    HoriPoint=HoriPart(img,Page,texts)[0]

    VertPoint=VertPart(path)[1]
    print('Failed detecting Vertical Point')
    HH,WW=img.shape[:2]
    VertPoint=1*HH//3
    
    img=cv2.imread(path)
    
    Show(PosiDict,Office,img,VertPoint,HoriPoint)


Pre10
議案課
Failed detecting Vertical Point
区政課
Failed detecting Vertical Point
報道課
Failed detecting Vertical Point
統計課
Failed detecting Vertical Point
Pre15
用品課
Failed detecting Vertical Point
購買課
Failed detecting Vertical Point
Pre20
動員課
Failed detecting Vertical Point
貯蓄奨励課
Failed detecting Vertical Point
商工課
Failed detecting Vertical Point
Pre25
體力課
Failed detecting Vertical Point
衛生課
Failed detecting Vertical Point
Pre30
Pre35
大久保病院
Failed detecting Vertical Point
大塚病院
Horizontal Line was not automatically detected... Defining line arbitrariry...
Failed detecting Vertical Point
Pre45
庶務課
Horizontal Line was not automatically detected... Defining line arbitrariry...
Failed detecting Vertical Point
計書課
Failed detecting Vertical Point
Pre50
治水課
Failed detecting Vertical Point
Pre55
葛飾区出張所
Failed detecting Vertical Point
江戸川区出張所
Horizontal Line was not automatically detected... Defining line arbitrariry...
Failed detecting Vertical Point
築港課
Horizontal Line was not automatically detecte

- Cause of Error
1. Horizontal Line inaccuracy: 4
2. Position not showing : 31
3. Inaccuracy in Office: 2| 庶務課，計書課, 調査課


In [21]:
CrossWalk=pd.read_csv("C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
Positions=CrossWalk.tolist()[:-1]
Positions        

['主事',
 '技師',
 '技師',
 '投手',
 '技手',
 '手',
 '事務員',
 '嘱託員',
 '書記',
 '収納書記',
 '社会書記',
 '臨時',
 '視学',
 '守衛長',
 '運轉手',
 '運輸手',
 '監督',
 '保健婦',
 '醫員',
 '警員',
 '調薬員',
 '監視',
 '講師',
 '鑑定員',
 '藥劑員',
 '看護婦長',
 '清掃監督',
 '船長',
 '機関手',
 '體力指導員',
 '區長']